In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import itertools
from sklearn.metrics.cluster import adjusted_mutual_info_score, adjusted_rand_score
from scipy import stats
from scipy.stats import pearsonr, kruskal, chi2_contingency, mannwhitneyu, linregress
import seaborn as sns
from ptitprince import PtitPrince as pt
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test
from lifelines.plotting import add_at_risk_counts
from sklearn.utils.class_weight import compute_class_weight
from pandas.api.types import is_numeric_dtype
from statsmodels.stats.multitest import multipletests
import statsmodels
from scipy.cluster.hierarchy import fcluster
import forestplot as fp
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import KBinsDiscretizer

In [ ]:
from general_functions import remove_small_clusters, clinical_enrichment, calculate_stability_metrics, consensus_matrix

In [ ]:
RANDOM_STATE = 42
n_clusters = 2

# Final clusters

To obtain the final clusters, the combination of miRNA and copy number (CNA) was used, and the algorithm MOFA applied. First, let's look at the stability of the clusters after n runs:

In [ ]:
sns.set_theme(style='ticks')

In [ ]:
import warnings
warnings.filterwarnings("ignore")
final_clusters = pd.read_csv('benchmarking_files/final_clusters_file.csv',
                             dtype={'view_combination': str},
                             converters={'y_pred': eval, 'y_pred_idx': eval, 
                                         'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
clinical_data_file = pd.read_csv('raw data/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)

In [ ]:
import warnings
warnings.filterwarnings("ignore")
final_clusters = pd.read_csv('benchmarks_new/final_clusters_file.csv',
                             dtype={'view_combination': str},
                             converters={'y_pred': eval, 'y_pred_idx': eval, 
                                         'relative_cluster_sizes': lambda x: eval(x.replace(': ', ':'))})
clinical_data_file = pd.read_csv('raw data/cancer_data_PAAD_clinical_data.tsv', sep='\t', header=0)

First, let's check if there are any very small clusters formed.

In [ ]:
valid_results, outlier_results, df_top_outliers = remove_small_clusters(final_clusters, 10, verbose=True)

In this case, there are no patients grouped into a small cluster.

In [ ]:
clin_results = clinical_enrichment(final_clusters, clinical_data_file)

In [ ]:
clin_results['pvalue_logrank'].mean()

In [ ]:
stability_dict = {}
for i in np.arange(10):
    subset = final_clusters.iloc[:(2**(i+1))]
    summarised_results = calculate_stability_metrics(subset, random_state=RANDOM_STATE, progress_bar=True)
    stability_dict[i] = summarised_results['AMI'].iloc[0]

In [ ]:
stability_df = pd.DataFrame(stability_dict.items(), columns=['log2(n_runs)', 'AMI'])
stability_df['n_runs'] = 2 ** (stability_df['log2(n_runs)'] + 1)
plt.figure(figsize=(6, 3))
sns.lineplot(x=stability_df['n_runs'], y=stability_df['AMI'])
plt.ylabel('Adj. Mutual Info. (AMI) score')
plt.ylim(0, 1)
plt.xlim(0, 1050)
plt.xlabel('Number of permutations')
plt.xscale('linear')
plt.savefig('figures/final_clusters/stability_finalclusters.svg', bbox_inches='tight')
plt.show()

In [ ]:
consensus_matrix = consensus_matrix(final_clusters)

In [ ]:
# Applying k-means clustering to consensus matrix
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE).fit(consensus_matrix)
kmeans_df = pd.DataFrame({
    'Patient ID': consensus_matrix.index,
    'Cluster': kmeans.labels_})
kmeans_clusters = {}
for cluster in range(n_clusters):
    kmeans_clusters[f'Cluster_{cluster}'] = kmeans_df[kmeans_df['Cluster'] == cluster]['Patient ID'].tolist()

In [ ]:
pd.DataFrame.from_dict(kmeans_clusters, orient='index').to_csv('patient_clusters.csv')

In [ ]:
# Applying hierarchical clustering to consensus matrix
cm = sns.clustermap(consensus_matrix, cmap='Blues', yticklabels=False, xticklabels=False)
linkage_matrix = cm.dendrogram_row.linkage
cluster_assignments = fcluster(linkage_matrix, t=n_clusters, criterion='maxclust')
plt.savefig('figures/final_clusters/consensus_final_clusters.svg', bbox_inches='tight')
plt.show()

patient_ids = consensus_matrix.index
hierarchical_clusters = {}
for cluster in range(1, n_clusters + 1):
    hierarchical_clusters[f'Cluster_{cluster-1}'] = [patient_ids[i] for i, assignment in enumerate(cluster_assignments) if assignment == cluster]

In [ ]:
# Function to compare hierarchical clusters with K-means clusters
hierarchical_sets = {key: set(values) for key, values in hierarchical_clusters.items()}
kmeans_sets = {key: set(values) for key,values in kmeans_clusters.items()}
for h_key, h_set in hierarchical_sets.items():
    match_found = False
    for k_key, k_set in kmeans_sets.items():
        if h_set == k_set:
            print(f"Hierarchical {h_key} matches K-means {k_key}")
            match_found = True
            break
    if not match_found:
        print(f"Hierarchical {h_key} does not match any K-means cluster")

In [ ]:
# Item consensus (only for two clusters)
def get_item_consensus(consensus_matrix: pd.DataFrame, ref_cluster_num: int):
    ref_cluster = hierarchical_clusters[f"Cluster_{ref_cluster_num}"]
    other_cluster = 0 if ref_cluster_num == 1 else 1
    item_consensus = {}
    for patient in consensus_matrix.columns:
        if patient in ref_cluster:
            cluster_matrix = consensus_matrix.loc[consensus_matrix.index.isin(ref_cluster), consensus_matrix.columns.isin(ref_cluster)]
            ref_cluster_avg = cluster_matrix[patient].mean()
            other_cluster_avg = 1 - ref_cluster_avg
        else:
            cluster_matrix = consensus_matrix.loc[~consensus_matrix.index.isin(ref_cluster), ~consensus_matrix.columns.isin(ref_cluster)]
            other_cluster_avg = cluster_matrix[patient].mean()
            ref_cluster_avg = 1 - other_cluster_avg
        item_consensus[patient] = {f"cluster{ref_cluster_num}": ref_cluster_avg, f"cluster{other_cluster}": other_cluster_avg}
    return item_consensus

In [ ]:
item_consensus = get_item_consensus(consensus_matrix, 1)
sorted_dict = dict(sorted(item_consensus.items(), key=lambda item: item[1]['cluster0'], reverse=True))

In [ ]:
cluster_0_patients = hierarchical_clusters['Cluster_0']
cluster_1_patients = hierarchical_clusters['Cluster_1']

probabilities = []
groups = []
for patient, probs in sorted_dict.items():
    # Use probability of belonging to cluster1, so that patients from cluster 0 have a probability of 0
    if patient in cluster_0_patients:
        probabilities.append(probs['cluster1'])  
        groups.append('Cluster 0')
    elif patient in cluster_1_patients:
        probabilities.append(probs['cluster1'])
        groups.append('Cluster 1')

df = pd.DataFrame({'Probability': probabilities, 'Cluster': groups})
plt.figure(figsize=(6, 3))
sns.kdeplot(data=df, x="Probability", palette='colorblind', hue="Cluster", 
            multiple="layer", fill=True, alpha=0.7)
palette = sns.color_palette("colorblind")
plt.legend(labels=['Cluster 1', 'Cluster 2'], loc='upper left', 
           handles=[plt.Rectangle((0, 0), 1, 1, color=palette[0], alpha=0.7), 
                    plt.Rectangle((0, 0), 1, 1, color=palette[1], alpha=0.7)])
plt.xlabel("Probability of belonging to Cluster 2")
plt.ylabel("Density")
plt.savefig('figures/final_clusters/item_consensus.svg', bbox_inches='tight')
plt.show()

# Clinical analysis

In [ ]:
clinical_data = clinical_data_file[['Patient ID', 'Mutation Count', 'Fraction Genome Altered', 'Diagnosis Age', 'Sex', 'Race Category', 
                                    'Adjuvant Postoperative Targeted Therapy Administered Indicator', 'Alcohol History Documented', 'Tumor resected max dimension',
                                    'American Joint Committee on Cancer Metastasis Stage Code', 'American Joint Committee on Cancer Tumor Stage Code',
                                    'Chronic Pancreatitis Personal Medical History Indicator', 'Did patient start adjuvant postoperative radiotherapy?', 
                                    'Disease Free Status', 'Family History of Cancer', 'Neoplasm Disease Lymph Node Stage American Joint Committee on Cancer Code', 
                                    'Neoplasm Disease Stage American Joint Committee on Cancer Code', 'Neoplasm Histologic Grade', 'TMB (nonsynonymous)',
                                    'New Neoplasm Event Post Initial Therapy Indicator', 'Overall Survival (Months)', 'Overall Survival Status', 'Disease Free (Months)', 
                                    'Participant Personal Medical History Diabetes Mellitus Ind-3', 'Patient Primary Tumor Site', 'Prior Cancer Diagnosis Occurence', 
                                    'Surgical Margin Resection Status', 'Patient Smoking History Category', 'Person Neoplasm Status', 'Primary Therapy Outcome Success Type']]
clinical_data['Overall Survival Status'] = clinical_data['Overall Survival Status'].str.split(':').str[0].astype(int)
clinical_data['Patient Smoking History Category'] = (clinical_data['Patient Smoking History Category']
                                                         .where(clinical_data['Patient Smoking History Category'].isna(), 
                                                                clinical_data['Patient Smoking History Category'].astype(float).astype(str)))
clinical_data.set_index('Patient ID', inplace=True)
clinical_data['Cluster'] = None
for cluster_num, patients in hierarchical_clusters.items():
    cluster_label = int(cluster_num.split('_')[-1])+1
    clinical_data.loc[clinical_data.index.isin(patients), 'Cluster'] = cluster_label
clinical_data.dropna(subset=['Cluster'], inplace=True)

### Survival curves

In [ ]:
# Check if weights are needed
test_weights_survival = clinical_data[['Overall Survival Status', 'Cluster']]
test_weights_survival['Cluster'] = pd.to_numeric(test_weights_survival['Cluster'])
result_weights_survival = linregress(x=test_weights_survival['Overall Survival Status'].values, y=test_weights_survival['Cluster'].values)
print(f"p = {result_weights_survival.pvalue}")

Since the p-value is < 0.05, we can assume that censoring through time isn't random, hence weights are needed

In [ ]:
survival_df = clinical_data[['Overall Survival Status', 'Overall Survival (Months)', 'Cluster']]
survival_df['Weights'] = None
for i in np.arange(min(survival_df['Overall Survival (Months)']), max(survival_df['Overall Survival (Months)']), 6):
    interval_df = survival_df[(survival_df['Overall Survival (Months)'] >= i) & (survival_df['Overall Survival (Months)'] < i + 6)].copy()
    censored_data = interval_df['Overall Survival Status'].to_numpy()
    unique_classes = np.unique(censored_data)
    if censored_data.size != 0:
        class_weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=censored_data)
        weight_mapping = dict(zip(unique_classes, class_weights))
        interval_df['Weights'] = interval_df['Overall Survival Status'].map(weight_mapping)
        survival_df.update(interval_df[['Weights']])
survival_df['Weights'] = pd.to_numeric(survival_df['Weights'])
survival_df.sort_values(by='Cluster', ascending=True, inplace=True)    # order to have analysis with respect to cluster 1

In [ ]:
colorblind_palette = sns.color_palette('colorblind')
plt.figure(figsize=(10, 5))
kmf_list = []
for cluster in survival_df['Cluster'].unique():
    cluster_data = survival_df[survival_df['Cluster'] == cluster]
    kmf = KaplanMeierFitter()
    # To ensure the plots show the patient events as they are (they change with weights), no weights are added for plots
    kmf.fit(cluster_data['Overall Survival (Months)'], cluster_data['Overall Survival Status'],
            label=f'Cluster {int(cluster)}')
    kmf.plot_survival_function(show_censors=True, color=colorblind_palette[cluster-1])
    kmf_list.append(kmf)
plt.text(x=0.1, y=0.25, s='Log-rank, p = 0.113')    # from R-code
plt.xlabel('Time (months)')
max_time = max(survival_df['Overall Survival (Months)'])
plt.xticks(list(np.arange(0, max_time+3, 6)))
plt.ylabel('Survival probability')
add_at_risk_counts(*kmf_list, ax=plt.gca())

cph_data = survival_df.reset_index(drop=True)
cph = CoxPHFitter()
cph_df = cph.fit(cph_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status', weights_col='Weights').summary
plt.text(x=0.1, y=0.15, s=f"HR = {cph_df['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_df['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_df['exp(coef) upper 95%'].loc['Cluster']:.2f})")

plt.text(x=0.1, y=0.05, s='RMST = 1.48, CI: (0.98, 2.28)')

plt.savefig('figures/final_clusters/survival_final_clusters.svg', bbox_inches='tight')
plt.show()

In [ ]:
colorblind_palette = sns.color_palette('colorblind')
plt.figure(figsize=(10, 5))
for cluster in survival_df['Cluster'].unique():
    cluster_data = survival_df[survival_df['Cluster'] == cluster]
    kmf = KaplanMeierFitter()
    # To ensure the plots show the patient events as they are (they change with weights), no weights are added for plots
    kmf.fit(cluster_data['Overall Survival (Months)'], cluster_data['Overall Survival Status'],
            label=f'Cluster {int(cluster)}')
    kmf.plot_survival_function(color=colorblind_palette[cluster-1], ci_show=False)
plt.xlabel('Time (months)')
max_time = max(survival_df['Overall Survival (Months)'])
plt.xticks(list(np.arange(0, max_time+3, 6)))
plt.ylabel('Survival probability')

plt.savefig('figures/final_clusters/survival_noci_final_clusters.svg', bbox_inches='tight')
plt.show()

In [ ]:
# Cox Proportional hazards, Fisher's test and RMST results at 6, 12, 36 and 60 months
months = [6, 12, 18, 24, 36, 60]
print('Hazard ratios')
for time in months: 
    print(f"{time} months:")
    data_surv = survival_df.copy()
    time_subset = survival_df['Overall Survival (Months)'] <= time
    data_surv.loc[~time_subset, 'Overall Survival Status'] = 0
    # Cox proportional hazards model
    cph_data_surv = data_surv.reset_index(drop=True)
    cph = CoxPHFitter()
    cph_surv = cph.fit(cph_data_surv, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
    print(f"Hazard Ratio = {cph_surv['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_surv['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_surv['exp(coef) upper 95%'].loc['Cluster']:.2f})")
    # Test if censoring is random
    data_surv['Cluster'] = pd.to_numeric(data_surv['Cluster'])
    test_censoring = linregress(x=data_surv['Overall Survival Status'].values, y=data_surv['Cluster'].values)
    print(f"Random censoring: p = {test_censoring.pvalue}")

### Disease free curves

In [ ]:
test_weights_disease_free = clinical_data[['Disease Free Status', 'Cluster']]
test_weights_disease_free = test_weights_disease_free.dropna(how='any',axis=0)
test_weights_disease_free['Cluster'] = pd.to_numeric(test_weights_disease_free['Cluster'])
test_weights_disease_free['Disease Free Status'] = test_weights_disease_free['Disease Free Status'].str.split(':').str[0].astype(int)
result_weights_disease_free = linregress(x=test_weights_disease_free['Disease Free Status'].values, y=test_weights_disease_free['Cluster'].values)
print(f"p = {result_weights_disease_free.pvalue}")

Since the p-value is > 0.05, we have insufficient evidence to reject the null hypothesis, hence assume the censoring was random, and don't need weights for future calculations.

In [ ]:
disease_free_df = clinical_data[['Disease Free Status', 'Disease Free (Months)', 'Cluster']]
disease_free_df.sort_values(by='Cluster', ascending=True, inplace=True)    # order to have analysis with respect to cluster 1
disease_free_df = disease_free_df.dropna(how='any',axis=0)
disease_free_df['Disease Free Status'] = disease_free_df['Disease Free Status'].str.split(':').str[0].astype(int)

In [ ]:
kmf_list = []
plt.figure(figsize=(10, 5))
for cluster in sorted(disease_free_df['Cluster'].unique()):
    cluster_data = disease_free_df[disease_free_df['Cluster'] == cluster]
    kmf = KaplanMeierFitter()
    kmf.fit(cluster_data['Disease Free (Months)'], cluster_data['Disease Free Status'],
            label=f'Cluster {int(cluster)}')
    kmf.plot_survival_function(show_censors=True, color=colorblind_palette[cluster-1])
    kmf_list.append(kmf)
max_time= max(disease_free_df['Disease Free (Months)'])
plt.xticks(list(np.arange(0, max_time+3, 6)))
plt.text(x=0.1, y=0.15, s= 'Log-rank, p = 0.216')    # from R-code
plt.xlabel('Time (months)')
plt.ylabel('Disease free probability')
add_at_risk_counts(*kmf_list, ax=plt.gca())

# RMST
plt.text(x=0.1, y=0.05, s='RMST = 1.09, CI: (0.76, 1.57)')

plt.savefig('figures/final_clusters/diseasefree_final_clusters.svg', bbox_inches='tight')
plt.show()

In [ ]:
kmf_list = []
plt.figure(figsize=(10, 5))
for cluster in sorted(disease_free_df['Cluster'].unique()):
    cluster_data = disease_free_df[disease_free_df['Cluster'] == cluster]
    kmf = KaplanMeierFitter()
    kmf.fit(cluster_data['Disease Free (Months)'], cluster_data['Disease Free Status'],
            label=f'Cluster {int(cluster)}')
    kmf.plot_survival_function(color=colorblind_palette[cluster-1], ci_show=False)
    kmf_list.append(kmf)
max_time= max(disease_free_df['Disease Free (Months)'])
plt.xticks(list(np.arange(0, max_time+3, 6)))
plt.xlabel('Time (months)')
plt.ylabel('Disease free probability')
plt.savefig('figures/final_clusters/rmst_diseasefree.svg', bbox_inches='tight')
plt.show()

In [ ]:
# Cox Proportional hazards results at 6, 12, 36 and 60 months
months = [6, 12, 18, 24, 36, 60]
for time in months: 
    print(f"{time} months:")
    data_diseasefree = disease_free_df.copy()
    time_subset = disease_free_df['Disease Free (Months)'] <= time
    data_diseasefree.loc[~time_subset, 'Disease Free Status'] = 0
    # Cox proportional hazards model
    cph_data_diseasefree = data_diseasefree.reset_index(drop=True)
    cph = CoxPHFitter()
    cph_diseasefree = cph.fit(cph_data_diseasefree, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
    print(f"Hazard Ratio = {cph_diseasefree['exp(coef)'].loc['Cluster']:.2f}, CI: ({cph_diseasefree['exp(coef) lower 95%'].loc['Cluster']:.2f}, {cph_diseasefree['exp(coef) upper 95%'].loc['Cluster']:.2f})")
    # Test if censoring is random
    data_diseasefree['Cluster'] = pd.to_numeric(data_diseasefree['Cluster'])
    test_censoring = linregress(x=data_diseasefree['Disease Free Status'].values, y=data_diseasefree['Cluster'].values)
    print(f"Random censoring: p = {test_censoring.pvalue}")

### Enrichment of clinical labels

In [ ]:
# Enrichment of clinical labels
clinical_labels = clinical_data.copy()
clinical_enrichment = {}
for variable in clinical_labels.columns:
    if pd.api.types.is_numeric_dtype(clinical_labels[variable]):
        test_numerical = [
            clinical_labels[clinical_labels['Cluster'] == cluster][variable].dropna().to_numpy() 
            for cluster in clinical_labels['Cluster'].unique()]
        stat, p_value_kruskal = kruskal(*test_numerical)
        clinical_enrichment[variable] = p_value_kruskal
    else: 
        test_discrete = pd.crosstab(clinical_labels['Cluster'], clinical_labels[variable])
        chi2, p_value_chi2, dof, expected = chi2_contingency(test_discrete)
        clinical_enrichment[variable] = p_value_chi2
del clinical_enrichment['Cluster'], clinical_enrichment['Overall Survival (Months)'], clinical_enrichment['Overall Survival Status']
clinical_enrichment_pvalues = pd.DataFrame.from_dict(clinical_enrichment, orient='index', columns=['Original p-value'])
reject, pvals_corr, asidack, abonf = statsmodels.stats.multitest.multipletests(pvals=clinical_enrichment_pvalues['Original p-value'], alpha=0.05, 
                                                                               method='fdr_bh', maxiter=1, is_sorted=False, returnsorted=False)
clinical_enrichment_pvalues['Adjusted p-value'] = pvals_corr
clinical_enrichment_pvalues['Significance'] = clinical_enrichment_pvalues['Adjusted p-value'].apply(lambda x: '*' if x < 0.05 else '')
clinical_enrichment_pvalues[clinical_enrichment_pvalues['Significance'] == '*']

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 3))
sns.boxplot(data=clinical_data, x='Cluster', y='Mutation Count', showmeans=True, palette='colorblind', ax=ax[0])
ax[0].text(x=0.2, y=0.9, s=f"p = {clinical_enrichment_pvalues['Adjusted p-value'].loc['Mutation Count']:.3e}", ha='center', va='top', transform=ax[0].transAxes)

sns.boxplot(data=clinical_data, x='Cluster', y='Fraction Genome Altered', showmeans=True, palette='colorblind', ax=ax[1])
ax[1].text(x=0.2, y=0.9, s=f"p = {clinical_enrichment_pvalues['Adjusted p-value'].loc['Fraction Genome Altered']:.3e}", ha='center', va='top', transform=ax[1].transAxes)
ax[1].set_ylabel('Fraction Genome Altered (%)')

sns.boxplot(data=clinical_data, x='Cluster', y='TMB (nonsynonymous)', showmeans=True, palette='colorblind', ax=ax[2])
ax[2].text(x=0.2, y=0.9, s=f"p = {clinical_enrichment_pvalues['Adjusted p-value'].loc['TMB (nonsynonymous)']:.3e}", ha='center', va='top', transform=ax[2].transAxes)

plt.tight_layout(w_pad=4)
plt.show()

There seems to be an extreme outlier in mutation count, so let's repeat the test after removing the outlier patient.

In [ ]:
outlier = clinical_data[clinical_data['Mutation Count'] > 500].index    # isolate the outlier from the plots
clinical_data_no_outlier = clinical_data.drop(outlier)
clinical_enrichment_no_outlier = {}
for variable in clinical_data_no_outlier.columns:
    if pd.api.types.is_numeric_dtype(clinical_data_no_outlier[variable]):
        test_numerical = [
            clinical_data_no_outlier[clinical_data_no_outlier['Cluster'] == cluster][variable].dropna().to_numpy() 
            for cluster in clinical_data_no_outlier['Cluster'].unique()]
        stat, p_value_kruskal = kruskal(*test_numerical)
        clinical_enrichment_no_outlier[variable] = p_value_kruskal
    else: 
        test_discrete = pd.crosstab(clinical_data_no_outlier['Cluster'], clinical_data_no_outlier[variable])
        chi2, p_value_chi2, dof, expected = chi2_contingency(test_discrete)
        clinical_enrichment_no_outlier[variable] = p_value_chi2
del clinical_enrichment_no_outlier['Cluster'], clinical_enrichment_no_outlier['Overall Survival (Months)'], clinical_enrichment_no_outlier['Overall Survival Status']
clinical_enrichment_pvalues_no_outlier = pd.DataFrame.from_dict(clinical_enrichment_no_outlier, orient='index', columns=['Original p-value'])
reject, pvals_corr, asidack, abonf = statsmodels.stats.multitest.multipletests(pvals=clinical_enrichment_pvalues_no_outlier['Original p-value'], alpha=0.05, 
                                                                               method='fdr_bh', maxiter=1, is_sorted=False, returnsorted=False)
clinical_enrichment_pvalues_no_outlier['Adjusted p-value'] = pvals_corr
clinical_enrichment_pvalues_no_outlier['Significance'] = clinical_enrichment_pvalues_no_outlier['Adjusted p-value'].apply(lambda x: '*' if x < 0.05 else '')
clinical_enrichment_pvalues_no_outlier[clinical_enrichment_pvalues_no_outlier['Significance'] == '*']

In [ ]:
from statannotations.Annotator import Annotator
pairs = [(1, 2)]
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
sns.boxplot(data=clinical_data_no_outlier, x='Cluster', y='Mutation Count', showmeans=True, palette='colorblind', ax=ax[0])
p_value_mutcount = clinical_enrichment_pvalues['Adjusted p-value'].loc['Mutation Count']
annotator_mutcount = Annotator(ax[0], pairs, data=clinical_data_no_outlier, x='Cluster', y='Mutation Count')
annotator_mutcount.configure(test=None, text_format="simple", loc="inside", verbose=2)
annotator_mutcount.set_custom_annotations([f"p = {p_value_mutcount:.2e}"])
annotator_mutcount.annotate()
ax[0].set_ylabel('Mutation Count')

sns.boxplot(data=clinical_data_no_outlier, x='Cluster', y='Fraction Genome Altered', showmeans=True, palette='colorblind', ax=ax[1])
p_value_fga = clinical_enrichment_pvalues['Adjusted p-value'].loc['Fraction Genome Altered']
annotator_fga = Annotator(ax[1], pairs, data=clinical_data_no_outlier, x='Cluster', y='Fraction Genome Altered')
annotator_fga.configure(test=None, text_format="simple", loc="inside", verbose=2)
annotator_fga.set_custom_annotations([f"p = {p_value_fga:.2e}"])
annotator_fga.annotate()
ax[1].set_ylabel('Fraction Genome Altered')

sns.boxplot(data=clinical_data_no_outlier, x='Cluster', y='TMB (nonsynonymous)', showmeans=True, palette='colorblind', ax=ax[2])
p_value_fga = clinical_enrichment_pvalues['Adjusted p-value'].loc['TMB (nonsynonymous)']
annotator_fga = Annotator(ax[2], pairs, data=clinical_data_no_outlier, x='Cluster', y='TMB (nonsynonymous)')
annotator_fga.configure(test=None, text_format="simple", loc="inside", verbose=2)
annotator_fga.set_custom_annotations([f"p = {p_value_fga:.2e}"])
annotator_fga.annotate()
ax[2].set_ylabel('Tumor Mutational Burden (nonsynonymous)')

plt.tight_layout(w_pad=4)
plt.savefig('figures/final_clusters/enriched_labels_boxplots.svg', bbox_inches='tight')
plt.show()

# Minimal biomarker panel

From the previous results, there are no differences in survival. However, the clusters are always very consistent, which could indicate that there is some underlying information that might be helpful or an indicator for pancreatic cancer. For that reason, we will check for biomarkers using a cross-validation and feature selection approach. Four methods will be used.

In [ ]:
survival_data = clinical_data[['Overall Survival Status', 'Overall Survival (Months)']]
recurrence_data = clinical_data[['Disease Free Status', 'Disease Free (Months)']]
recurrence_data.dropna(inplace=True)
recurrence_data['Disease Free Status'] = recurrence_data['Disease Free Status'].str.split(':').str[0].astype(int)

### ABESS analysis

In [ ]:
from abess import LogisticRegression
from contextlib import contextmanager
from sklearn.base import BaseEstimator, TransformerMixin
import copy

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
random_numbers = np.random.randint(0, 100 + 1, size=10).tolist()
months = [6, 12, 18, 24, 36, 60]

In [ ]:
@contextmanager
def fixed_seed(seed):
    state = np.random.get_state()
    np.random.seed(seed)
    try:
        yield
    finally:
        np.random.set_state(state)
        
class ABESSClassif(LogisticRegression):
    def __init__(self, p=None, random_state=None):
        self.p = p
        self.random_state = random_state
        with fixed_seed(self.random_state):
            super().__init__(support_size=p)

    def fit(self, X, y=None):
        with fixed_seed(self.random_state):
            super().fit(X=X, y=y)
        coef = self.coef_
        if coef.ndim == 1:
            coef = coef[np.newaxis, :]
        features = X.columns[np.where(np.any(coef != 0, axis=0))[0]]
        self.features_ = list(set(features))
        if self.p is not None:
            assert self.p == len(self.features_)
        return self

    def transform(self, X):
        return X[self.features_]

class MultiModalIFABESS(BaseEstimator, TransformerMixin):
    def __init__(self, random_state=None):
        self.random_state = random_state

    def fit(self, Xs, y):
        self.models_per_view_ = []
        self.selected_columns_per_view_ = []
        for X in Xs:
            model = ABESSClassif(random_state=self.random_state)
            model.fit(X, y)
            coef = model.coef_.ravel()
            support = np.where(coef != 0)[0]
            selected_cols = X.columns[support].tolist()
            self.models_per_view_.append(model)
            self.selected_columns_per_view_.append(selected_cols)
        selected_Xs = []
        for X, cols in zip(Xs, self.selected_columns_per_view_):
            selected_Xs.append(X[cols])
        X_combined = pd.concat(selected_Xs, axis=1)
        self.final_model_ = ABESSClassif(random_state=self.random_state)
        self.final_model_.fit(X_combined, y)
        coef_final = self.final_model_.coef_.ravel()
        mask = np.where(coef_final != 0)[0]
        self.features_ = X_combined.columns[mask].tolist()
        return self

    def transform(self, Xs):
        selected_Xs = []
        for X, cols in zip(Xs, self.selected_columns_per_view_):
            selected_Xs.append(X[cols])
        X_combined = pd.concat(selected_Xs, axis=1)
        return X_combined[self.features_]


class SplitMultiModalWrapper(BaseEstimator, TransformerMixin):
    def __init__(self, selector, cna_cols, methyl_cols):
        self.selector = selector
        self.cna_cols = cna_cols
        self.methyl_cols = methyl_cols

    def fit(self, X, y=None):
        X_cna = X[self.cna_cols]
        X_methyl = X[self.methyl_cols]
        X_list = [X_cna, X_methyl]
        self.selector.fit(X_list, y)
        return self

    def transform(self, X):
        X_cna = X[self.cna_cols]
        X_methyl = X[self.methyl_cols]
        X_list = [X_cna, X_methyl]        
        return self.selector.transform(X_list)

In [ ]:
# METHYLATION FEATURE SELECTION
methylation_data = pd.read_csv('processed_data/methylation_processed.csv', index_col=0)
methylation_data['Cluster'] = None
for cluster_num, patients in hierarchical_clusters.items():
    cluster_label = int(cluster_num.split('_')[-1])
    methylation_data.loc[methylation_data.index.isin(patients), 'Cluster'] = cluster_label
methylation_data.dropna(subset=['Cluster'], inplace=True)
X_methyl = methylation_data.drop(columns='Cluster').astype(float)
y_methyl = methylation_data['Cluster'].astype(int)

# Matthews coefficient (all features)
crossval_methyl = cross_val_score(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'), 
                                  X_methyl, y_methyl, cv=skf, scoring='matthews_corrcoef')
print("MCC score: %0.4f, standard deviation: %0.4f" % (crossval_methyl.mean(), crossval_methyl.std()))

# Do ABESS on methylation features
pipeline_methyl = make_pipeline(ABESSClassif(random_state=RANDOM_STATE),
                                RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample', oob_score=True))
scores_methyl = cross_val_score(pipeline_methyl, X_methyl, y_methyl, cv=skf, scoring='matthews_corrcoef')
pipeline_methyl.fit(X_methyl, y_methyl)
features_selected_methyl = pipeline_methyl.named_steps['abessclassif'].features_
print(f"Selected methylation features: {features_selected_methyl}")
print(f"Matthews Correlation Coefficient: {scores_methyl.mean():.4f}, standard deviation: {scores_methyl.std():.4f}")
print(f"Out-of-bag (OOB) score: {pipeline_methyl.named_steps['randomforestclassifier'].oob_score_:.4f}")

features_selected_methyl = pipeline_methyl.named_steps['abessclassif'].features_
selected_methyl_data = X_methyl[features_selected_methyl]

# SURVIVAL PLOTS
cox_methyl_survival = selected_methyl_data.merge(survival_data, left_index=True, right_index=True)
print("PLOTS (SURVIVAL):")
fig, axes = plt.subplots(1, len(features_selected_methyl), figsize=(6 * len(features_selected_methyl), 2))
print("Overall:")
for i, col_idx in enumerate(features_selected_methyl):
    biomarker = col_idx
    biomarker_data = cox_methyl_survival[[biomarker, 'Overall Survival (Months)', 'Overall Survival Status']]
    cph = CoxPHFitter()
    summary = cph.fit(biomarker_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
    print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
    summary.reset_index(inplace=True)
    ax = axes[i] if len(features_selected_methyl) > 1 else axes
    fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
        varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
plt.tight_layout()
plt.show()
for time in months:
    print(f"{time} months:")
    cox_copy = cox_methyl_survival.copy()
    time_subset = cox_copy['Overall Survival (Months)'] <= time
    cox_copy.loc[~time_subset, 'Overall Survival Status'] = 0
    fig, axes = plt.subplots(1, len(features_selected_methyl), figsize=(6 * len(features_selected_methyl), 2))
    for i, col_idx in enumerate(features_selected_methyl):
        biomarker = col_idx
        biomarker_data = cox_copy[[biomarker, 'Overall Survival (Months)', 'Overall Survival Status']]
        cph = CoxPHFitter()
        summary = cph.fit(biomarker_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
        print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
        summary.reset_index(inplace=True)
        ax = axes[i] if len(features_selected_methyl) > 1 else axes
        fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
            varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
    plt.tight_layout()
    plt.show()

# RECURRENCE PLOTS
cox_methyl_recurrence = selected_methyl_data.merge(recurrence_data, left_index=True, right_index=True)
print("PLOTS (RECURRENCE):")
fig, axes = plt.subplots(1, len(features_selected_methyl), figsize=(6 * len(features_selected_methyl), 2))
print("Overall:")
for i, col_idx in enumerate(features_selected_methyl):
    biomarker = col_idx
    biomarker_data = cox_methyl_recurrence[[biomarker, 'Disease Free (Months)', 'Disease Free Status']]
    cph = CoxPHFitter()
    summary = cph.fit(biomarker_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
    print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
    summary.reset_index(inplace=True)
    ax = axes[i] if len(features_selected_methyl) > 1 else axes
    fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
        varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
plt.tight_layout()
plt.show()
for time in months:
    print(f"{time} months:")
    cox_copy = cox_methyl_recurrence.copy()
    time_subset = cox_copy['Disease Free (Months)'] <= time
    cox_copy.loc[~time_subset, 'Disease Free Status'] = 0
    fig, axes = plt.subplots(1, len(features_selected_methyl), figsize=(6 * len(features_selected_methyl), 2))
    for i, col_idx in enumerate(features_selected_methyl):
        biomarker = col_idx
        biomarker_data = cox_copy[[biomarker, 'Disease Free (Months)', 'Disease Free Status']]
        cph = CoxPHFitter()
        summary = cph.fit(biomarker_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
        print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
        summary.reset_index(inplace=True)
        ax = axes[i] if len(features_selected_methyl) > 1 else axes
        fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
            varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
    plt.tight_layout()
    plt.show()

In [ ]:
# COPY NUMBER SURVIVAL FEATURES
cna_data = pd.read_csv('processed_data/CNA_processed.csv', index_col=0)
cna_data['Cluster'] = None
for cluster_num, patients in hierarchical_clusters.items():
    cluster_label = int(cluster_num.split('_')[-1])
    cna_data.loc[cna_data.index.isin(patients), 'Cluster'] = cluster_label
cna_data.dropna(subset=['Cluster'], inplace=True)
X_cna = cna_data.drop(columns='Cluster').astype(float)
y_cna = cna_data['Cluster'].astype(int)

# Matthews coefficient (all features)
crossval_cna = cross_val_score(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'), 
                               X_cna, y_cna, cv=skf, scoring='matthews_corrcoef')
print("MCC score: %0.4f, standard deviation: %0.4f" % (crossval_cna.mean(), crossval_cna.std()))

# Do ABESS on copy number features
pipeline_cna = make_pipeline(ABESSClassif(random_state=RANDOM_STATE), 
                             RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample', oob_score=True))
scores_cna = cross_val_score(pipeline_cna, X_cna, y_cna, cv=skf, scoring='matthews_corrcoef')
pipeline_cna.fit(X_cna, y_cna)
features_selected_cna = pipeline_cna.named_steps['abessclassif'].features_
print(f"Selected copy number features: {features_selected_cna}")
print(f"Matthews Correlation Coefficient: {scores_cna.mean():.4f}, standard deviation: {scores_cna.std():.4f}")
print(f"Out-of-bag (OOB) score: {pipeline_cna.named_steps['randomforestclassifier'].oob_score_:.4f}")

features_selected_cna = pipeline_cna.named_steps['abessclassif'].features_
selected_cna_data = X_cna[features_selected_cna]

# SURVIVAL PLOTS
print("PLOTS (SURVIVAL):")
cox_cna_survival = selected_cna_data.merge(survival_data, left_index=True, right_index=True)
print("Overall:")
fig, axes = plt.subplots(1, len(features_selected_cna), figsize=(6 * len(features_selected_cna), 2))
for i, col_idx in enumerate(features_selected_cna):
    biomarker = col_idx
    biomarker_data = cox_cna_survival[[biomarker, 'Overall Survival (Months)', 'Overall Survival Status']]
    cph = CoxPHFitter()
    summary = cph.fit(biomarker_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
    print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
    summary.reset_index(inplace=True)
    ax = axes[i] if len(features_selected_cna) > 1 else axes
    fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
        varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
plt.tight_layout()
plt.show()
for time in months:
    print(f"{time} months:")
    cox_copy = cox_cna_survival.copy()
    time_subset = cox_copy['Overall Survival (Months)'] <= time
    cox_copy.loc[~time_subset, 'Overall Survival Status'] = 0
    fig, axes = plt.subplots(1, len(features_selected_cna), figsize=(6 * len(features_selected_cna), 2))
    for i, col_idx in enumerate(features_selected_cna):
        biomarker = col_idx
        biomarker_data = cox_copy[[biomarker, 'Overall Survival (Months)', 'Overall Survival Status']]
        cph = CoxPHFitter()
        summary = cph.fit(biomarker_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
        print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
        summary.reset_index(inplace=True)
        ax = axes[i] if len(features_selected_cna) > 1 else axes
        fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
            varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
    plt.tight_layout()
    plt.show()

# RECURRENCE PLOTS
print("PLOTS (RECURRENCE):")
cox_cna_recurrence = selected_cna_data.merge(recurrence_data, left_index=True, right_index=True)
fig, axes = plt.subplots(1, len(features_selected_cna), figsize=(6 * len(features_selected_cna), 2))
print("Overall:")
for i, col_idx in enumerate(features_selected_cna):
    biomarker = col_idx
    biomarker_data = cox_cna_recurrence[[biomarker, 'Disease Free (Months)', 'Disease Free Status']]
    cph = CoxPHFitter()
    summary = cph.fit(biomarker_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
    print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
    summary.reset_index(inplace=True)
    ax = axes[i] if len(features_selected_cna) > 1 else axes
    fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
        varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
plt.tight_layout()
plt.show()
for time in months:
    print(f"{time} months:")
    cox_copy = cox_cna_recurrence.copy()
    time_subset = cox_copy['Disease Free (Months)'] <= time
    cox_copy.loc[~time_subset, 'Disease Free Status'] = 0
    fig, axes = plt.subplots(1, len(features_selected_cna), figsize=(6 * len(features_selected_cna), 2))
    for i, col_idx in enumerate(features_selected_cna):
        biomarker = col_idx
        biomarker_data = cox_copy[[biomarker, 'Disease Free (Months)', 'Disease Free Status']]
        cph = CoxPHFitter()
        summary = cph.fit(biomarker_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
        print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
        summary.reset_index(inplace=True)
        ax = axes[i] if len(features_selected_cna) > 1 else axes
        fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
            varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
    plt.tight_layout()
    plt.show()

In [ ]:
# ABESS WITH INTERMEDIATE FEATURE SELECTION
X_cna = pd.read_csv("processed_data/CNA_processed.csv", index_col=0)
X_methylation = pd.read_csv("processed_data/methylation_processed.csv", index_col=0)
combined_data = pd.concat([X_methylation, X_cna], axis=1)
combined_data.dropna(inplace=True)
combined_data['Cluster'] = None
for cluster_num, patients in hierarchical_clusters.items():
    cluster_label = int(cluster_num.split('_')[-1])
    combined_data.loc[combined_data.index.isin(patients), 'Cluster'] = cluster_label
# combined_data.dropna(axis=0, inplace=True)
X_total = combined_data.drop(columns='Cluster').dropna(axis=0).astype(float)
y_total = combined_data.loc[X_total.index, 'Cluster'].astype(int)

# Matthews coefficient (all features)
crossval_total = cross_val_score(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'), 
                                 X_total, y_total, cv=skf, scoring='matthews_corrcoef')
print("MCC score: %0.4f, standard deviation: %0.4f" % (crossval_total.mean(), crossval_total.std()))

common_index = X_cna.index.intersection(X_methylation.index).intersection(y_total.index)
X_cna = X_cna.loc[common_index]
X_methylation = X_methylation.loc[common_index]
cna_cols = X_cna.columns
methyl_cols = X_methylation.columns
X_combined = pd.concat([X_cna, X_methylation], axis=1)

selector = MultiModalIFABESS(random_state = RANDOM_STATE)
wrapped_selector = SplitMultiModalWrapper(selector, cna_cols, methyl_cols)
pipeline_multi = make_pipeline(wrapped_selector,
                               RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample', oob_score=True))
scores_multi = cross_val_score(pipeline_multi, X_combined, y_total, cv=skf, scoring='matthews_corrcoef')
pipeline_multi.fit(X_combined, y_total)
features_selected_final = pipeline_multi.named_steps['splitmultimodalwrapper'].selector.features_
print(f"Selected multimodal features: {features_selected_final}")
print(f"Matthews Correlation Coefficient: {scores_multi.mean():.4f}, standard deviation: {scores_multi.std():.4f}")
print(f"Out-of-bag (OOB) score: {pipeline_multi.named_steps['randomforestclassifier'].oob_score_:.4f}")

# SURVIVAL PLOTS
selected_final_data = X_combined[features_selected_final]
cox_input_survival = selected_final_data.merge(survival_data, left_index=True, right_index=True)
print("PLOTS (SURVIVAL):")
print("Overall:")
fig, axes = plt.subplots(1, len(features_selected_final), figsize=(6 * len(features_selected_final), 2))
for i, col_idx in enumerate(features_selected_final):
    biomarker = col_idx
    biomarker_data = cox_input_survival[[biomarker, 'Overall Survival (Months)', 'Overall Survival Status']]
    cph = CoxPHFitter()
    summary = cph.fit(biomarker_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
    print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
    summary.reset_index(inplace=True)
    ax = axes[i] if len(features_selected_final) > 1 else axes
    fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
        varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
plt.tight_layout()
plt.show()
for time in months:
    print(f"{time} months:")
    cox_copy = cox_input_survival.copy()
    time_subset = cox_copy['Overall Survival (Months)'] <= time
    cox_copy.loc[~time_subset, 'Overall Survival Status'] = 0
    fig, axes = plt.subplots(1, len(features_selected_final), figsize=(6 * len(features_selected_final), 2))
    for i, col_idx in enumerate(features_selected_final):
        biomarker = col_idx
        biomarker_data = cox_copy[[biomarker, 'Overall Survival (Months)', 'Overall Survival Status']]
        cph = CoxPHFitter()
        summary = cph.fit(biomarker_data, duration_col='Overall Survival (Months)', event_col='Overall Survival Status').summary
        print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
        summary.reset_index(inplace=True)
        ax = axes[i] if len(features_selected_final) > 1 else axes
        fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
            varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
    plt.tight_layout()
    plt.show()

# RECURRENCE PLOTS
cox_input_recurrence = selected_final_data.merge(recurrence_data, left_index=True, right_index=True)
print("PLOTS (RECURRENCE):")
print("Overall:")
fig, axes = plt.subplots(1, len(features_selected_final), figsize=(6 * len(features_selected_final), 2))
for i, col_idx in enumerate(features_selected_final):
    biomarker = col_idx
    biomarker_data = cox_input_recurrence[[biomarker, 'Disease Free (Months)', 'Disease Free Status']]
    cph = CoxPHFitter()
    summary = cph.fit(biomarker_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
    print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
    summary.reset_index(inplace=True)
    ax = axes[i] if len(features_selected_final) > 1 else axes
    fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
        varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
plt.tight_layout()
plt.show()
for time in months:
    print(f"{time} months:")
    cox_copy = cox_input_recurrence.copy()
    time_subset = cox_copy['Disease Free (Months)'] <= time
    cox_copy.loc[~time_subset, 'Disease Free Status'] = 0
    fig, axes = plt.subplots(1, len(features_selected_final), figsize=(6 * len(features_selected_final), 2))
    for i, col_idx in enumerate(features_selected_final):
        biomarker = col_idx
        biomarker_data = cox_copy[[biomarker, 'Disease Free (Months)', 'Disease Free Status']]
        cph = CoxPHFitter()
        summary = cph.fit(biomarker_data, duration_col='Disease Free (Months)', event_col='Disease Free Status').summary
        print(f"{biomarker}: p-val = {summary['p'].loc[biomarker]:.2f}, HR = {summary['exp(coef)'].loc[biomarker]:.2f}")
        summary.reset_index(inplace=True)
        ax = axes[i] if len(features_selected_final) > 1 else axes
        fp.forestplot(summary, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%',
            varlabel='covariate', pval='p', color_alt_rows=True, ax=ax, title=biomarker, xticks=np.linspace(0.1, 10, 5))
    plt.tight_layout()
    plt.show()

### Differential methylation analysis

In [ ]:
methylation_data = pd.read_csv('processed_data/methylation_processed.csv', index_col=0)
methylation_data['Cluster'] = None
for cluster_num, patients in hierarchical_clusters.items():
    cluster_label = int(cluster_num.split('_')[-1])
    methylation_data.loc[methylation_data.index.isin(patients), 'Cluster'] = cluster_label
methylation_data.dropna(subset=['Cluster'], inplace=True)

In [ ]:
# Calculate M values (better for statistical analyses)
mvalues_df = methylation_data.copy()
columns_to_transform = mvalues_df.columns[:-1]
mvalues_df[columns_to_transform] = mvalues_df[columns_to_transform].applymap(lambda x: x/(1-x))
methyl_cluster0 = mvalues_df[mvalues_df['Cluster'] == 0]
methyl_cluster1 = mvalues_df[mvalues_df['Cluster'] == 1]
avg_methyl_values = pd.concat([methyl_cluster0.mean(axis=0), methyl_cluster1.mean(axis=0)], axis=1)
avg_methyl_values.rename(columns={0: 'Cluster0', 1: 'Cluster1'}, inplace=True)
avg_methyl_values.drop('Cluster', inplace=True)
avg_methyl_values['Delta(M)'] = avg_methyl_values['Cluster1'] - avg_methyl_values['Cluster0']

# Calculate FDR using Benjamini-Hochberg correction for multiple tests
from scipy.stats import ttest_ind
p_values = []
columns_to_test = methyl_cluster0.columns[:-1]
for column in columns_to_test:
    values_cluster0 = methyl_cluster0[column]
    values_cluster1 = methyl_cluster1[column]
    t_stat, p_val = ttest_ind(values_cluster0, values_cluster1, equal_var=False)
    p_values.append(p_val)
avg_methyl_values['p-value'] = p_values
_, fdr_values, _, _ = statsmodels.stats.multitest.multipletests(avg_methyl_values['p-value'], method='fdr_bh', is_sorted=False, returnsorted=False)
avg_methyl_values['FDR'] = fdr_values
methyl_mvalues = avg_methyl_values.copy()

In [ ]:
# Plot volcano plot for differential methylation expression
plt.figure(figsize=(8, 6))
a = 10**-8
diff_methyl_sites = ((methyl_mvalues['Delta(M)'] <= -2) | (methyl_mvalues['Delta(M)'] >= 2)) & (-np.log10(methyl_mvalues['FDR']) > -np.log10(a))
colors = np.where(diff_methyl_sites, 'red', 'grey')
plt.scatter(x=methyl_mvalues['Delta(M)'], y=-np.log10(methyl_mvalues['FDR']), c=colors, alpha=0.7)
plt.axhline(y=-np.log10(a), color='gray', linestyle='--', linewidth=1)
plt.axvline(x=-2, color='gray', linestyle='--', linewidth=1)
plt.axvline(x=2, color='gray', linestyle='--', linewidth=1)
plt.xlabel('Delta(M)')
plt.ylabel('-log10(FDR)')
plt.xlim(-28, 10)
from adjustText import adjust_text
texts = []
for i, row in methyl_mvalues[diff_methyl_sites].iterrows():
    x = row['Delta(M)']
    y = -np.log10(row['FDR'])
    texts.append(plt.text(x, y, str(i), color='black'))
adjust_text(texts, arrowprops=dict(arrowstyle='-', color='black', lw=0.5))
plt.savefig('figures/final_clusters/volcano_methyl.svg', bbox_inches='tight')
plt.show()

### Feature selection for copy number

In [ ]:
cna_data = pd.read_csv('processed_data/CNA_processed.csv', index_col=0)
cna_data['Cluster'] = None
for cluster_num, patients in hierarchical_clusters.items():
    cluster_label = int(cluster_num.split('_')[-1])
    cna_data.loc[cna_data.index.isin(patients), 'Cluster'] = cluster_label
cna_data.dropna(subset=['Cluster'], inplace=True)

In [ ]:
X = cna_data.iloc[:, :-1].to_numpy(dtype=float)
y = cna_data.iloc[:, -1].to_numpy(dtype=int)
feature_names = cna_data.iloc[:, :-1].columns.to_numpy()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
crossval_bw = cross_val_score(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'), 
                              X, y, cv=skf, scoring='matthews_corrcoef')
print("MCC score: %0.2f, standard deviation: %0.2f" % (crossval_bw.mean(), crossval_bw.std()))

In [ ]:
pipeline_bw = make_pipeline(SequentialFeatureSelector(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'), 
                                                      tol=0.01, n_jobs=6, cv=skf, scoring='matthews_corrcoef', direction='forward'), 
                            RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample', oob_score=True))
scores_bw = cross_val_score(pipeline_bw, X, y, cv=skf, scoring='matthews_corrcoef')
pipeline_bw.fit(X, y)
biomarkers_bw = feature_names[pipeline_bw.named_steps['sequentialfeatureselector'].get_support()].tolist()
print(f"Features selected: {biomarkers_bw}")
print(f"Matthews Correlation Coefficient: {scores_bw.mean():.3f}, standard deviation: {scores_bw.std():.3f}")
print(f"Out-of-bag (OOB) score: {pipeline_bw.named_steps['randomforestclassifier'].oob_score_}")

In [ ]:
biomarkers_data_bw = cna_data[biomarkers_bw]
patients_data = survival_df[['Overall Survival Status', 'Overall Survival (Months)']]
biomarkers_survival_bw = biomarkers_data_bw.merge(patients_data, left_index=True, right_index=True)

fig, ax= plt.subplots(1, 2, figsize=(20, 4), width_ratios=[0.3, 0.7])
cph = CoxPHFitter()
bw_biomarkers_cph = cph.fit(biomarkers_survival_bw, 'Overall Survival (Months)', 'Overall Survival Status').summary
bw_biomarkers_cph.reset_index(inplace=True)
fp.forestplot(bw_biomarkers_cph, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-3, 3.5, 1), ax=ax[0])

boxplots_df_bw = biomarkers_survival_bw.merge(cna_data['Cluster'], left_index=True, right_index=True)
boxplots_df_bw_melted = boxplots_df_bw.melt(id_vars="Cluster", value_vars=biomarkers_bw,
                                                  var_name="Variable", value_name="Expression")
pt.RainCloud(x="Variable", y="Expression", hue="Cluster", data=boxplots_df_bw_melted, orient="v", 
             width_viol=0.6, alpha=0.7, move=0.2, palette='colorblind', dodge=True, ax=ax[1])
ax[1].set_xlabel('Biomarker')
ax[1].set_ylim(0, 1.2)
plt.tight_layout()
plt.savefig('figures/biomarkers_bw.svg', bbox_inches='tight')
plt.show()

for biomarker in biomarkers_bw:
    group0 = boxplots_df_bw[boxplots_df_bw['Cluster'] == 0][biomarker]
    group1 = boxplots_df_bw[boxplots_df_bw['Cluster'] == 1][biomarker]
    stat, p_value = mannwhitneyu(group0, group1, alternative='two-sided')
    print(f"{biomarker} p-value: {p_value}")

In [ ]:
biomarkers_data_bw = cna_data[biomarkers_bw]
patients_data_diseasefree = disease_free_df[['Disease Free Status', 'Disease Free (Months)']]
biomarkers_diseasefree_bw = biomarkers_data_bw.merge(patients_data_diseasefree, left_index=True, right_index=True)

fig, ax= plt.subplots(1, 2, figsize=(20, 4), width_ratios=[0.3, 0.7])
cph = CoxPHFitter()
bw_biomarkers_cph_diseasefree = cph.fit(biomarkers_diseasefree_bw, 'Disease Free (Months)', 'Disease Free Status').summary
bw_biomarkers_cph_diseasefree.reset_index(inplace=True)
fp.forestplot(bw_biomarkers_cph_diseasefree, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-5, 7, 1), ax=ax[0])

boxplots_df_bw_diseasefree = biomarkers_diseasefree_bw.merge(cna_data['Cluster'], left_index=True, right_index=True)
boxplots_df_bw_melted_diseasefree = boxplots_df_bw_diseasefree.melt(id_vars="Cluster", value_vars=biomarkers_bw,
                                                                      var_name="Variable", value_name="Expression")
pt.RainCloud(x="Variable", y="Expression", hue="Cluster", data=boxplots_df_bw_melted_diseasefree, orient="v", 
             width_viol=0.6, alpha=0.7, move=0.2, palette='colorblind', dodge=True, ax=ax[1])
ax[1].set_xlabel('Biomarker')
ax[1].set_ylim(0, 1.2)
plt.tight_layout()
plt.savefig('figures/biomarkers_bw_diseasefree.svg', bbox_inches='tight')
plt.show()

for biomarker in biomarkers_bw:
    group0 = boxplots_df_bw_diseasefree[boxplots_df_bw_diseasefree['Cluster'] == 0][biomarker]
    group1 = boxplots_df_bw_diseasefree[boxplots_df_bw_diseasefree['Cluster'] == 1][biomarker]
    stat, p_value = mannwhitneyu(group0, group1, alternative='two-sided')
    print(f"{biomarker} p-value: {p_value}")

In [ ]:
features_selected_bw = biomarkers_bw.copy()
cna_data_bw = cna_data.copy()
i = 1
while True:
    cna_data_bw = cna_data_bw.drop(columns=features_selected_bw)
    X_bw = cna_data_bw.iloc[:, :-1].to_numpy(dtype=float)
    y_bw = cna_data_bw.iloc[:, -1].to_numpy(dtype=int)
    feature_names_bw = cna_data_bw.iloc[:, :-1].columns.to_numpy()
    print(f"Run {i}")
    print('Cross validation (all features):')
    new_crossval_bw = cross_val_score(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'),
                                       X_bw, y_bw, cv=skf, scoring='matthews_corrcoef')
    print("MCC score: %0.2f, standard deviation: %0.2f" % (new_crossval_bw.mean(), new_crossval_bw.std()))
    
    print('Feature selection:')
    new_pipeline_bw = make_pipeline(SequentialFeatureSelector(RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample'), 
                                                               tol=0.01, n_jobs=6, cv=skf, scoring='matthews_corrcoef', direction='forward'), 
                                     RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced_subsample', oob_score=True))
    new_scores_bw = cross_val_score(new_pipeline_bw, X_bw, y_bw, cv=skf, scoring='matthews_corrcoef')
    new_pipeline_bw.fit(X_bw, y_bw)
    features_selected_bw = feature_names_bw[new_pipeline_bw.named_steps['sequentialfeatureselector'].get_support()].tolist()
    print(f"Features selected: {features_selected_bw}")
    print(f"Matthews Correlation Coefficient: {new_scores_bw.mean():.3f}, standard deviation: {new_scores_bw.std():.3f}")
    print(f"Out-of-bag (OOB) score: {new_pipeline_bw.named_steps['randomforestclassifier'].oob_score_}")
    if (scores_bw.mean()) - (new_scores_bw.mean()) > 0.05:
        break
    i += 1

In [ ]:
# Test biomarkers individually (survival)
all_biomarkers = biomarkers_bw.copy()
fig, ax = plt.subplots(2, 4, figsize=(18, 5))
ax = ax.flatten()
# First plot: forest plot for all common biomarkers
biomarkers_data = cna_data[all_biomarkers]
patients_data = survival_df[['Overall Survival Status', 'Overall Survival (Months)']]
biomarkers_survival = biomarkers_data.merge(patients_data, left_index=True, right_index=True)
cph = CoxPHFitter()
biomarkers_cph = cph.fit(biomarkers_survival, 'Overall Survival (Months)', 'Overall Survival Status').summary
biomarkers_cph.reset_index(inplace=True)
fp.forestplot(biomarkers_cph, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-1, 2.5, 1), ax=ax[0])
# Other plots: individual plots for all common biomarkers
for i, biomarker in enumerate(all_biomarkers):
    biomarkers_data = cna_data[[biomarker]]
    patients_data = survival_df[['Overall Survival Status', 'Overall Survival (Months)', 'Weights']]
    biomarkers_survival = biomarkers_data.merge(patients_data, left_index=True, right_index=True)
    cph = CoxPHFitter()
    biomarkers_cph = cph.fit(biomarkers_survival, 'Overall Survival (Months)', 'Overall Survival Status', weights_col='Weights').summary
    biomarkers_cph.reset_index(inplace=True)
    fp.forestplot(biomarkers_cph, estimate='exp(coef)', ll='exp(coef) lower 95%', hl='exp(coef) upper 95%', varlabel='covariate',
                  pval='p', color_alt_rows=True, xticks=np.arange(-1, 2, 1), ax=ax[i+1])   
plt.subplots_adjust(wspace=3, hspace=0.5)
plt.show()

In [ ]:
# Test biomarkers individually (disease-free)
# all_biomarkers = list(set(biomarkers_rfc + biomarkers_est + biomarkers_bw))
fig, ax = plt.subplots(2, 4, figsize=(18, 5))
ax = ax.flatten()

biomarkers_data = cna_data[all_biomarkers]
patients_data = disease_free_df[['Disease Free Status', 'Disease Free (Months)']]
biomarkers_diseasefree = biomarkers_data.merge(patients_data, left_index=True, right_index=True)
cph = CoxPHFitter()
biomarkers_cph = cph.fit(biomarkers_diseasefree, 'Disease Free (Months)', 'Disease Free Status').summary
biomarkers_cph.reset_index(inplace=True)
fp.forestplot(biomarkers_cph, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate', 
              pval='p', color_alt_rows=True, figsize=(4,4), xticks=np.arange(-1, 2.5, 1), ax=ax[0])

for i, biomarker in enumerate(all_biomarkers):
    biomarkers_data = cna_data[[biomarker]]
    patients_data = disease_free_df[['Disease Free Status', 'Disease Free (Months)']]
    biomarkers_diseasefree = biomarkers_data.merge(patients_data, left_index=True, right_index=True)
    cph = CoxPHFitter()
    biomarkers_cph = cph.fit(biomarkers_diseasefree, 'Disease Free (Months)', 'Disease Free Status').summary
    biomarkers_cph.reset_index(inplace=True)
    fp.forestplot(biomarkers_cph, estimate='coef', ll='coef lower 95%', hl='coef upper 95%', varlabel='covariate',
                  pval='p', color_alt_rows=True, xticks=np.arange(-1, 2, 1), ax=ax[i+1])   
plt.subplots_adjust(wspace=3, hspace=0.5)
plt.show()

In [ ]:
amp_data = pd.read_csv('raw data/data_gistic_genes_amp.txt', sep="	", index_col=0)
del_data = pd.read_csv('raw data/data_gistic_genes_del.txt', sep="	", index_col=0)
gene_data_gistic = pd.concat([amp_data, del_data])
gene_data_gistic.set_index('chromosome', drop=True, inplace=True)
gene_data_gistic.sort_values(by='chromosome', inplace=True)
filtered_data_gistic = gene_data_gistic[gene_data_gistic['cytoband'].isin(all_biomarkers)]
genes_cna = filtered_data_gistic['genes_in_peak'].tolist()
genes_cna = [gene for sublist in genes_cna for gene in sublist.split(',') if gene]
filtered_data_gistic

### Heatmaps for methylation and CNA data, with cluster assignments

In [ ]:
hierarchical_cluster0 = hierarchical_clusters['Cluster_0']
hierarchical_cluster1 = hierarchical_clusters['Cluster_1']

In [ ]:
# Get data from both methylation and CNA
CNA_file = pd.read_csv('processed_data/CNA_processed.csv', index_col=0)
CNA_file['Cluster'] = None
for cluster_num, patients in hierarchical_clusters.items():
    cluster_label = int(cluster_num.split('_')[-1])
    CNA_file.loc[CNA_file.index.isin(patients), 'Cluster'] = cluster_label
CNA_data = CNA_file.sort_values(by='Cluster')

methylation_file = pd.read_csv('processed_data/methylation_processed.csv', index_col=0)
methylation_file['Cluster'] = None
for cluster_num, patients in hierarchical_clusters.items():
    cluster_label = int(cluster_num.split('_')[-1])
    methylation_file.loc[methylation_file.index.isin(patients), 'Cluster'] = cluster_label
methylation_data = methylation_file.sort_values(by='Cluster')

# Plot heatmaps
clusters_CNA = CNA_data['Cluster'].unique()
palette_CNA = sns.color_palette("colorblind", len(clusters_CNA))
color_mapping_CNA = dict(zip(clusters_CNA, palette_CNA))
cluster_colors_CNA = CNA_data['Cluster'].map(color_mapping_CNA)
cm_cna = sns.clustermap(data=CNA_data.drop(columns=['Cluster']).T, col_colors=cluster_colors_CNA, cmap='coolwarm', col_cluster=False, 
                        row_cluster=False, figsize=(8,5), center=0, xticklabels=False, yticklabels=False, cbar_pos=(0.1, 0.2, 0.02, 0.4))
cm_cna.ax_heatmap.set_xlabel('Patients')
cm_cna.ax_heatmap.set_ylabel('Cytobands')
plt.savefig('figures/final_clusters/cna_heatmap.png', bbox_inches='tight')
plt.show()
clusters_methylation = methylation_data['Cluster'].unique()
palette_methylation = sns.color_palette("colorblind", len(clusters_methylation))
color_mapping_methylation = dict(zip(clusters_methylation, palette_methylation))
cluster_colors_methylation = methylation_data['Cluster'].map(color_mapping_methylation)
cm_methyl = sns.clustermap(data=methylation_data.drop(columns=['Cluster']).T, col_colors=cluster_colors_methylation, cmap='Blues', col_cluster=False, 
                          row_cluster=False, figsize=(8, 5),xticklabels=False, yticklabels=False, cbar_pos=(0.1, 0.2, 0.02, 0.4), vmin=0, vmax=1)
cm_methyl.ax_heatmap.set_xlabel('Patients')
cm_methyl.ax_heatmap.set_ylabel('Methylation sites')
plt.savefig('figures/final_clusters/methyl_heatmap.png', bbox_inches='tight')
plt.show()